In [69]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
# allow importing the "backbone" package when the kernel's cwd is notebooks/
sys.path.append(os.path.abspath(".."))
import backbone.custom_metrics.AstroMLmod3 as AstroMLmod
import backbone.custom_metrics.Intrinsic_dimension as id_module
from backbone.data_handle.Test import KNN_accuracy
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import davies_bouldin_score

# Galaxy10: TPCF, ID and quality metrics per epoch (Supervised vs BYOL vs DINO)

In [ ]:
import importlib
from sklearn.decomposition import IncrementalPCA
importlib.reload(AstroMLmod)

DATA_DIR = "/idia/projects/camil/Koketso/data_"
PLOTS_DIR = "/idia/projects/camil/Koketso/plots"
epochs = range(0, 100, 5)

# per-epoch Galaxy10 test/val representations: supervised pickles (representations, labels), BYOL/DINO CSVs (Image_names, label, features)
runs = {
    "Supervised": f"{PLOTS_DIR}/supervised_galaxy10_no_perturb_test_representations_labels{{epoch}}.csv",
    "BYOL": f"{DATA_DIR}/byol_galaxy10_val_representations_epoch_{{epoch}}.csv",
    "DINO": f"{DATA_DIR}/dino_galaxy10_val_representations_epoch_{{epoch}}.csv",
}

rows = []
for method, path_template in runs.items():
    for epoch in epochs:
        path = path_template.format(epoch=epoch)
        if not os.path.exists(path):
            continue
        if method == "Supervised":
            with open(path, "rb") as file:
                representations, labels = pickle.load(file)
            representations, labels = np.array(representations), list(labels)
        else:
            df = pd.read_csv(path)
            representations, labels = df.drop(columns=["Image_names", "label"]).to_numpy(), df["label"].tolist()
        n = representations.shape[0]

        # TPCF_score bootstraps over subsamples and returns (mean, std)
        tpcf, tpcf_std = AstroMLmod.TPCF_score(representations, epoch=epoch)
        id_mean, id_std = id_module.id_score(representations)
        db_score = davies_bouldin_score(representations, labels)
        knn_acc, knn_acc_std = KNN_accuracy(representations, labels)

        # average pairwise cosine similarity, excluding self-similarity on the diagonal
        cos_sim_matrix = cosine_similarity(representations)
        off_diag_mask = ~np.eye(n, dtype=bool)
        avg_cos_sim = cos_sim_matrix[off_diag_mask].mean()
        std_cos_sim = cos_sim_matrix[off_diag_mask].std()

        # anisotropy: fraction of total variance explained by the top principal component,
        # on unit-normalised vectors so a single extreme-norm point cannot dominate the variance
        unit_representations = representations / np.linalg.norm(representations, axis=1, keepdims=True)
        anisotropy = IncrementalPCA(n_components=1).fit(unit_representations).explained_variance_ratio_[0]

        rows.append({
            "method": method,
            "epoch": epoch,
            "tpcf_score": tpcf,
            "tpcf_std": tpcf_std,
            "id_mean": float(np.ravel(id_mean)[0]),
            "id_std": float(np.ravel(id_std)[0]),
            "avg_cosine_similarity": avg_cos_sim,
            "cosine_similarity_std": std_cos_sim,
            "anisotropy": anisotropy,
            "davies_bouldin": db_score,
            "knn_accuracy": knn_acc,
            "knn_accuracy_std": knn_acc_std,
        })
        print(f"{method} epoch {epoch}: TPCF={tpcf:.4f}±{tpcf_std:.4f} ID={rows[-1]['id_mean']:.2f} CosineSim={avg_cos_sim:.4f}±{std_cos_sim:.4f} Anisotropy={anisotropy:.4f} DaviesBouldin={db_score:.4f} KNNAcc={knn_acc:.2f}±{knn_acc_std:.2f}")

In [ ]:
results = pd.DataFrame(rows)
results.to_csv(f"{PLOTS_DIR}/galaxy10_metrics_per_epoch.csv", index=False)
results

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import AutoMinorLocator
import string

plt.rcParams.update({
    "figure.dpi": 300,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",
    "font.size": 11,
    "axes.labelsize": 12,
    "axes.titlesize": 12,
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "grid.linewidth": 0.5,
    "lines.linewidth": 1.5,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "xtick.minor.visible": True,
    "ytick.minor.visible": True,
    "xtick.top": True,
    "ytick.right": True,
    "legend.frameon": False,
})

ACCENT = "#1f4e79"
# fixed categorical order: colour follows the method
COLORS = {"Supervised": "#2a78d6", "BYOL": "#eb6834", "DINO": "#1baf7a"}

metrics = [
    ("tpcf_score", "TPCF score", "tpcf_std"),
    ("id_mean", "Intrinsic dimension", "id_std"),
    ("avg_cosine_similarity", "Avg. cosine similarity", "cosine_similarity_std"),
    ("anisotropy", "Anisotropy (top-1 EVR)", None),
    ("davies_bouldin", "Davies-Bouldin score", None),
    ("knn_accuracy", "KNN accuracy (%)", "knn_accuracy_std"),
]


def plot_gradient_band(ax, x, y, y_err, color=ACCENT, n_layers=40, max_sigma=3.0):
    """Shade the uncertainty band with an alpha gradient that fades away from the mean."""
    x = np.asarray(x)
    y = np.asarray(y)
    y_err = np.asarray(y_err)
    sigmas = np.linspace(max_sigma / n_layers, max_sigma, n_layers)
    for sigma in sigmas[::-1]:
        alpha = 0.35 * np.exp(-0.5 * sigma ** 2)
        ax.fill_between(x, y - sigma * y_err, y + sigma * y_err, color=color, alpha=alpha, linewidth=0)


n_cols = 3
n_rows = -(-len(metrics) // n_cols)  # ceil division

fig, axes = plt.subplots(
    n_rows, n_cols, figsize=(3.3 * n_cols, 2.5 * n_rows), squeeze=False, constrained_layout=True
)
axes = axes.flatten()

for panel_label, ax, (col, title, err_col) in zip(string.ascii_lowercase, axes, metrics):
    for method, data in results.groupby("method", sort=False):
        color = COLORS[method]
        ax.plot(data["epoch"], data[col], color=color, label=method)
        if col == "tpcf_score":
            plot_gradient_band(ax, data["epoch"], data[col], data[err_col], color=color)
        elif err_col is not None:
            ax.fill_between(
                data["epoch"],
                data[col] - data[err_col],
                data[col] + data[err_col],
                color=color,
                alpha=0.2,
            )
    ax.set_title(title)
    ax.set_xlabel("Epoch")
    ax.set_ylabel(title)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.text(-0.15, 1.08, f"({panel_label})", transform=ax.transAxes, fontsize=11, fontweight="bold")

fig.legend(*axes[0].get_legend_handles_labels(), loc="outside upper center", ncol=len(COLORS))

# hide any unused axes in the grid
for ax in axes[len(metrics):]:
    ax.axis("off")

fig.savefig(f"{PLOTS_DIR}/galaxy10_metrics_per_epoch_grid.pdf", bbox_inches="tight")
fig.savefig(f"{PLOTS_DIR}/galaxy10_metrics_per_epoch_grid.png", bbox_inches="tight")
plt.show()